# 07 — Outliers + Correlación — Airbnb Listings

## Setup

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

try:
    from scipy import stats
    SCIPY = True
except ImportError:
    SCIPY = False
    print('scipy no instalado — se usa implementación manual del z-score')

def find_project_root():
    for parent in [Path.cwd()] + list(Path.cwd().parents):
        if (parent / 'data').exists():
            return parent

ROOT = find_project_root()
df = pd.read_csv(ROOT / 'data' / 'external' / 'Listings.csv',
                 encoding='latin-1', low_memory=False)

# Tipos correctos antes de cualquier análisis
df['host_since'] = pd.to_datetime(df['host_since'])
for col in ['host_is_superhost', 'host_has_profile_pic',
            'host_identity_verified', 'instant_bookable']:
    df[col] = df[col].map({'t': True, 'f': False})

print(f'Shape: {df.shape}')
print(f'price — min: {df["price"].min()}, max: {df["price"].max():,}, media: {df["price"].mean():.0f}')


scipy no instalado — se usa implementación manual del z-score
Shape: (279712, 33)
price — min: 0, max: 625,216, media: 609


---
# Bloque 1 — Outliers

Un outlier es un valor que se aleja significativamente del resto. Puede ser un error de datos, un caso real extremo, o algo representativo del negocio. El método detecta — la decisión es contextual.

## Diagnóstico previo — distribución de `price`

Antes de aplicar ningún método, ver cómo está distribuida la columna. La forma de la distribución determina qué método es más adecuado.

In [2]:
print(df['price'].describe().round(2))
print(f'\nAsimetría (skewness): {df["price"].skew():.2f}')
# Skewness > 1 o < -1 indica distribución asimétrica
# En ese caso IQR es más robusto que z-score
print(f'Valores en 0: {(df["price"] == 0).sum()}')
# Un precio de 0 es casi siempre un error de datos


count    279712.00
mean        608.79
std        3441.83
min           0.00
25%          75.00
50%         150.00
75%         474.00
max      625216.00
Name: price, dtype: float64

Asimetría (skewness): 61.62
Valores en 0: 113


In [3]:
fig = px.histogram(df[df['price'] < 1000], x='price', nbins=80,
                   title='Distribución de precio por noche (excluye >1000 para visibilidad)',
                   labels={'price': 'Precio (USD/noche)'})
fig.show()


## Método 1 — IQR

Mejor para distribuciones asimétricas (la mayoría de variables de negocio). No asume ninguna forma de distribución. Los límites se calculan desde los cuartiles, no desde la media.

In [5]:
Q1  = df['price'].quantile(0.25)
Q3  = df['price'].quantile(0.75)
IQR = Q3 - Q1

# El factor 1.5 es el estándar de Tukey — cubre ~99.3% de una distribución normal
# Con 3.0 se detectan solo outliers extremos
limite_inf = Q1 - 1.5 * IQR
limite_sup = Q3 + 1.5 * IQR

outliers_iqr = df[(df['price'] < limite_inf) | (df['price'] > limite_sup)]

print(f'Q1={Q1}, Q3={Q3}, IQR={IQR}')
print(f'Límite inferior: {limite_inf:.0f}')
print(f'Límite superior: {limite_sup:.0f}')
print(f'\nOutliers IQR: {len(outliers_iqr):,} filas ({len(outliers_iqr)/len(df)*100:.1f}%)')


Q1=75.0, Q3=474.0, IQR=399.0
Límite inferior: -524
Límite superior: 1072

Outliers IQR: 31,395 filas (11.2%)


## Método 2 — Z-score

Mide cuántas desviaciones estándar se aleja un valor de la media. El umbral estándar es 3 (cubre el 99.7% de una distribución normal). Menos robusto cuando la distribución es asimétrica porque la media y la desviación estándar se distorsionan con los propios outliers.

In [6]:
if SCIPY:
    df['zscore_price'] = stats.zscore(df['price'].dropna())
else:
    # Implementación manual: (valor - media) / desviación estándar
    media = df['price'].mean()
    std   = df['price'].std()
    df['zscore_price'] = (df['price'] - media) / std

outliers_z = df[df['zscore_price'].abs() > 3]

print(f'Outliers z-score (|z| > 3): {len(outliers_z):,} filas ({len(outliers_z)/len(df)*100:.1f}%)')
print(f'\nComparación:')
print(f'  IQR detectó:     {len(outliers_iqr):>6,} outliers')
print(f'  Z-score detectó: {len(outliers_z):>6,} outliers')

# Los dos métodos casi nunca dan el mismo número
# IQR suele detectar más en distribuciones asimétricas


Outliers z-score (|z| > 3): 1,173 filas (0.4%)

Comparación:
  IQR detectó:     31,395 outliers
  Z-score detectó:  1,173 outliers


## Visualizar: boxplot + distribución

In [8]:
fig = go.Figure()

fig.add_trace(go.Box(
    y=df['price'],
    name='price',
    boxpoints='outliers',   # muestra solo los puntos outlier, no todos
    marker_color='steelblue',
    line_color='steelblue'
))

# Líneas de límites IQR
for valor, label in [(limite_sup, 'Límite IQR sup'), (limite_inf, 'Límite IQR inf')]:
    fig.add_hline(y=valor, line_dash='dash', line_color='tomato',
                  annotation_text=label, annotation_position='right')

fig.update_layout(title='Boxplot de precio por noche — límites IQR marcados',
                  yaxis_title='Precio (USD)')
fig.show()


## `minimum_nights` — outliers con decisión contextual

Esta columna tiene outliers legítimos (alquileres de larga estancia) y posibles errores (valores extremos como 365, 1125). El contexto de negocio es lo que determina qué hacer con cada caso.

In [9]:
print(df['minimum_nights'].describe().round(1))
print(f'\nDistribución de valores extremos:')
print(df['minimum_nights'].value_counts().head(10))


count    279712.0
mean          8.1
std          31.5
min           1.0
25%           1.0
50%           2.0
75%           5.0
max        9999.0
Name: minimum_nights, dtype: float64

Distribución de valores extremos:
minimum_nights
1     92839
2     60877
3     37869
30    28881
5     13770
4     12856
7     11356
6      3471
10     2810
14     2330
Name: count, dtype: int64


In [10]:
Q1_mn  = df['minimum_nights'].quantile(0.25)
Q3_mn  = df['minimum_nights'].quantile(0.75)
IQR_mn = Q3_mn - Q1_mn
lim_mn = Q3_mn + 1.5 * IQR_mn

extremos = df[df['minimum_nights'] > lim_mn]
print(f'Límite IQR superior: {lim_mn:.0f} noches')
print(f'Outliers detectados: {len(extremos):,} ({len(extremos)/len(df)*100:.1f}%)')
print()
print('Decisión según contexto:')
print('  minimum_nights == 1125 → probable error de entrada de datos → eliminar')
print('  minimum_nights == 365  → alquiler anual legítimo → conservar y anotar')
print('  minimum_nights == 30   → alquiler mensual → conservar, puede ser subgrupo')
print()
# Mostrar cuántos tienen valores > 365 (claramente erróneos)
print(f'Listados con >365 noches mínimas: {(df["minimum_nights"] > 365).sum()}')


Límite IQR superior: 11 noches
Outliers detectados: 43,105 (15.4%)

Decisión según contexto:
  minimum_nights == 1125 → probable error de entrada de datos → eliminar
  minimum_nights == 365  → alquiler anual legítimo → conservar y anotar
  minimum_nights == 30   → alquiler mensual → conservar, puede ser subgrupo

Listados con >365 noches mínimas: 96


---
# Bloque 2 — Correlación

La correlación mide la relación lineal entre dos variables numéricas. Va de -1 a 1.
- `1.0` — relación perfecta positiva (cuando una sube, la otra sube igual)
- `-1.0` — relación perfecta negativa (cuando una sube, la otra baja igual)
- `0.0` — sin relación lineal

Correlación alta entre dos variables en un modelo indica que están aportando la misma información.

## Selección de columnas numéricas relevantes

In [11]:
cols_num = [
    'price',
    'accommodates',
    'bedrooms',
    'minimum_nights',
    'review_scores_rating',
    'review_scores_cleanliness',
    'host_total_listings_count',
]

# Solo las columnas que existen en el DataFrame
cols_num = [c for c in cols_num if c in df.columns]

df_num = df[cols_num].dropna()
print(f'Filas con todas las columnas completas: {len(df_num):,}')
print(df_num.describe().round(2))


Filas con todas las columnas completas: 168,149
           price  accommodates   bedrooms  minimum_nights  \
count  168149.00     168149.00  168149.00       168149.00   
mean      497.00          3.46       1.50            7.07   
std      2147.76          2.13       1.03           33.29   
min         8.00          1.00       1.00            1.00   
25%        75.00          2.00       1.00            1.00   
50%       146.00          3.00       1.00            2.00   
75%       421.00          4.00       2.00            4.00   
max    300177.00         16.00      50.00         9999.00   

       review_scores_rating  review_scores_cleanliness  \
count             168149.00                  168149.00   
mean                  93.58                       9.33   
std                    9.93                       1.13   
min                   20.00                       2.00   
25%                   92.00                       9.00   
50%                   96.00                      10.00

## Matriz de correlación

In [1]:
corr = df_num.corr().round(2)
print(corr)


NameError: name 'df_num' is not defined

## Heatmap — máscara triangular

La matriz de correlación es simétrica: el valor de A↔B es el mismo que B↔A. Se muestra solo el triángulo inferior para evitar la redundancia visual.

In [13]:
# Máscara: poner NaN en el triángulo superior para que no se muestre
corr_masked = corr.copy().astype(float)
for i in range(len(corr_masked)):
    for j in range(i + 1, len(corr_masked.columns)):
        corr_masked.iloc[i, j] = float('nan')

fig = px.imshow(
    corr_masked,
    text_auto='.2f',
    color_continuous_scale='RdBu_r',   # rojo = negativo, azul = positivo
    zmin=-1, zmax=1,
    title='Correlación entre métricas de listing'
)
fig.update_layout(coloraxis_colorbar_title='r')
fig.show()


## Interpretar los resultados

In [15]:
print('Pares con correlación alta (|r| > 0.5):')
print('─' * 45)

# Iterar solo el triángulo inferior para no duplicar pares
for i in range(len(corr.columns)):
    for j in range(i):
        r = corr.iloc[i, j]
        if abs(r) > 0.5:
            col_a = corr.columns[i]
            col_b = corr.columns[j]
            direccion = 'positiva' if r > 0 else 'negativa'
            print(f'{col_a} ↔ {col_b}: r={r:.2f} ({direccion})')

print()
print('Pares sin relación (|r| < 0.1):')
print('─' * 45)
for i in range(len(corr.columns)):
    for j in range(i):
        r = corr.iloc[i, j]
        if abs(r) < 0.1:
            print(f'{corr.columns[i]} ↔ {corr.columns[j]}: r={r:.2f}')


Pares con correlación alta (|r| > 0.5):
─────────────────────────────────────────────
bedrooms ↔ accommodates: r=0.68 (positiva)
review_scores_cleanliness ↔ review_scores_rating: r=0.74 (positiva)

Pares sin relación (|r| < 0.1):
─────────────────────────────────────────────
minimum_nights ↔ price: r=-0.01
minimum_nights ↔ accommodates: r=-0.03
minimum_nights ↔ bedrooms: r=-0.02
review_scores_rating ↔ price: r=0.01
review_scores_rating ↔ accommodates: r=-0.00
review_scores_rating ↔ bedrooms: r=0.01
review_scores_rating ↔ minimum_nights: r=-0.00
review_scores_cleanliness ↔ price: r=0.02
review_scores_cleanliness ↔ accommodates: r=-0.00
review_scores_cleanliness ↔ bedrooms: r=0.00
review_scores_cleanliness ↔ minimum_nights: r=-0.02
host_total_listings_count ↔ price: r=0.01
host_total_listings_count ↔ accommodates: r=0.03
host_total_listings_count ↔ bedrooms: r=0.02
host_total_listings_count ↔ minimum_nights: r=0.00
host_total_listings_count ↔ review_scores_rating: r=-0.04
host_total_list

## Correlación ≠ causalidad

`r` mide relación lineal, no causa. Dos cosas pueden correlacionar por una tercera variable que no está en el análisis.

| r | Interpretación habitual |
|---|-------------------------|
| 0.9 – 1.0 | Muy alta — probable redundancia entre variables |
| 0.7 – 0.9 | Alta |
| 0.5 – 0.7 | Moderada |
| 0.3 – 0.5 | Débil |
| < 0.3 | Sin relación lineal relevante |

En modelado: si dos features tienen `r > 0.85`, eliminar una de las dos antes de entrenar.

---
## Resumen de decisiones sobre outliers

| Caso | Decisión | Razón |
|------|----------|-------|
| Precio = 0 | Eliminar | Error de datos — ningún listing es gratuito |
| Precio = 50,000 | Investigar | Puede ser un error o un alquiler de lujo real |
| minimum_nights > 365 | Eliminar | Imposible alquilar más noches de las que tiene un año |
| minimum_nights = 365 | Conservar + anotar | Alquiler anual legítimo, subgrupo distinto |
| review_score = 1.0 | Conservar | Puntuación válida aunque extrema |
